# 05 — GRU Sentiment Model

Train the GRU model with the shared tokenizer and split files. Keep the model settings aligned with the other recurrent model.

In [ ]:
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
import json
import time
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from src.config import PROCESSED_DIR, MODEL_DIR, METRICS_DIR, RANDOM_SEED, CLASS_NAMES, NUM_CLASSES, MAX_VOCAB_SIZE, EMBEDDING_DIM, RECURRENT_UNITS, DROPOUT_RATE, DENSE_UNITS, BATCH_SIZE, MAX_EPOCHS, LEARNING_RATE
from src.sequence_data import load_tokenizer, load_sequence_config, texts_to_padded_sequences
from src.models import build_recurrent_model

MODEL_TYPE = "gru"
tf.keras.utils.set_random_seed(RANDOM_SEED)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)
train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
val_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
tokenizer = load_tokenizer(PROCESSED_DIR / "tokenizer.json")
sequence_config = load_sequence_config(PROCESSED_DIR / "sequence_config.json")
max_length = int(sequence_config["max_length"])


## 1. Prepare identical sequence inputs

Both recurrent models must load the same tokenizer JSON, sequence length, and split manifests.

In [ ]:
X_train = texts_to_padded_sequences(train_df["review_clean"].astype(str), tokenizer, max_length)
X_val = texts_to_padded_sequences(val_df["review_clean"].astype(str), tokenizer, max_length)
y_train = train_df["label_id"].astype("int64").to_numpy()
y_val = val_df["label_id"].astype("int64").to_numpy()
vocab_size = min(MAX_VOCAB_SIZE, len(tokenizer.word_index) + 1)
print("Train:", X_train.shape, "Validation:", X_val.shape)
print("Vocabulary size:", vocab_size, "Max length:", max_length)


## 2. Build model and calculate training-only class weights

In [ ]:
model = build_recurrent_model(
    model_type=MODEL_TYPE, vocab_size=vocab_size, max_length=max_length,
    num_classes=NUM_CLASSES, embedding_dim=EMBEDDING_DIM,
    recurrent_units=RECURRENT_UNITS, dense_units=DENSE_UNITS,
    dropout_rate=DROPOUT_RATE, learning_rate=LEARNING_RATE,
)
model.summary()
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight = {int(label): float(weight) for label, weight in zip(classes, weights)}
print("Training class weights:", class_weight)


## 3. Train with early stopping

The checkpoint is selected by validation loss. Do not inspect test performance during training.

In [ ]:
model_path = MODEL_DIR / f"{MODEL_TYPE}_best.keras"
callbacks = [
    EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
    ModelCheckpoint(model_path, monitor="val_loss", save_best_only=True, verbose=1),
]
start_time = time.perf_counter()
history = model.fit(
    X_train, y_train, validation_data=(X_val, y_val),
    epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
    class_weight=class_weight, callbacks=callbacks, verbose=1,
)
training_seconds = time.perf_counter() - start_time
history_df = pd.DataFrame(history.history)
history_df.to_csv(METRICS_DIR / f"{MODEL_TYPE}_training_history.csv", index=False)
config = {
    "model_type": MODEL_TYPE, "seed": RANDOM_SEED, "vocab_size": vocab_size,
    "max_length": max_length, "embedding_dim": EMBEDDING_DIM,
    "recurrent_units": RECURRENT_UNITS, "dense_units": DENSE_UNITS,
    "dropout_rate": DROPOUT_RATE, "batch_size": BATCH_SIZE,
    "max_epochs": MAX_EPOCHS, "learning_rate": LEARNING_RATE,
    "training_seconds": training_seconds, "class_weight": class_weight,
    "best_model_path": str(model_path.relative_to(PROJECT_ROOT)),
}
(METRICS_DIR / f"{MODEL_TYPE}_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)
print(f"Training time: {training_seconds:.2f} seconds")
print("Saved best model:", model_path)


## 4. Validation-only diagnostic

This section is for checking training behaviour. Final test evaluation is performed in notebook 06 after both models and the baseline are ready.

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(7, 4))
plt.plot(history.history["loss"], label="train_loss")
plt.plot(history.history["val_loss"], label="val_loss")
plt.title(f"{MODEL_TYPE.upper()} Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.show()
